# Scraping y APIs con Python

Ejecuta las celdas de arriba abajo. Si no puedes acceder a las webs, ve al último apartado: «Lectura de muestras locales».

El web scraping extrae datos del HTML de una web. Verás cómo leer una tabla de hockey con pandas y cómo extraer nombres y precios de un catálogo de frutas con Beautiful Soup. Para consultar una API, usarás las publicaciones de JSONPlaceholder.

### Diferencias entre HTML y JSON

|  | Scraping de una tabla | Consulta de una API |
| --- | --- | --- |
| Recurso consultado | Página de estadísticas de hockey. | Listado de publicaciones: `/posts`. |
| Formato de respuesta | HTML: texto que describe la página y contiene la tabla. | JSON: lista de registros con campos y valores. |
| Conversión a tabla | Lectura de la tabla HTML con pandas. | Lectura del JSON y creación de un DataFrame. |

### Antes de consultar una web

Lee las indicaciones de rastreo del sitio, identifica lo que devuelve el servidor y elige cómo leerlo. Los tres apartados siguientes explican `robots.txt`, las peticiones HTTP y el formato JSON.

### Primera celda del cuaderno

In [ ]:
from io import StringIO
import pandas as pd
import requests

Qué hace cada línea:

1. `from io import StringIO` importa una herramienta de Python que permite pasar a pandas el texto HTML como si fuera un archivo.
2. `import pandas as pd` importa pandas y le da el alias `pd`. Se encargará de construir los DataFrames.
3. `import requests` importa la biblioteca que hará las peticiones HTTP.

Ejecuta las celdas en orden. Si Colab indica que falta el analizador `lxml` al leer la tabla HTML, instálalo en una celda con `%pip install lxml` y vuelve a ejecutar la lectura. El cuaderno incluye muestras locales para trabajar sin consultar las webs.

## A. robots.txt: instrucciones de rastreo

Un rastreador es un programa que visita páginas automáticamente. El archivo `robots.txt` contiene instrucciones sobre qué rutas puede rastrear. Si existe, se consulta en la raíz del sitio: se mantiene el dominio y se sustituye toda la ruta por `/robots.txt`.

[Consultar robots.txt de Scrape This Site](https://www.scrapethissite.com/robots.txt)

### Cómo se leen las reglas

```text
User-agent: *
Disallow: /lessons/
Disallow: /faq/
```

| Línea | Interpretación |
| --- | --- |
| `User-agent: *` | Grupo aplicable a los rastreadores que no tienen un grupo más específico. |
| `Disallow: /lessons/` | Pide evitar las rutas que empiezan por `/lessons/`. |
| `Disallow: /faq/` | Pide evitar las rutas que empiezan por `/faq/`, incluidas las páginas que cuelgan de esa ruta. |

`Allow` se puede usar para permitir una ruta. Al coincidir varias reglas del mismo grupo, se aplica la ruta más específica. En el fragmento anterior solo aparecen reglas `Disallow`.

### Aplicación a la tabla de hockey

La ruta `/pages/forms/` no empieza por `/lessons/` ni por `/faq/`, así que ninguna de esas dos reglas la excluye. La página `/faq/` se puede abrir manualmente en el navegador para consultar el permiso que da el sitio.

[Preguntas frecuentes de Scrape This Site](https://www.scrapethissite.com/faq/)

`robots.txt` no bloquea por sí solo el acceso ni sustituye a las condiciones de uso. Si la dirección del archivo devuelve `404` («no encontrado»), el archivo no está disponible ahí. Eso no da permiso por sí solo para extraer o reutilizar datos.

### Comprueba la lectura

¿Cuál de estas rutas coincide con un `Disallow` del fragmento: `/pages/forms/` o `/lessons/intro/`? Justifica la respuesta fijándote en el principio de cada ruta.

### ✏️ Respuesta

Coincide `/lessons/intro/`, porque empieza por `/lessons/`, que está en un `Disallow`.

`/pages/forms/` no empieza por ninguna ruta prohibida, así que ninguna regla la excluye.

## B.1. HTTP: peticiones y respuestas

HTTP es el protocolo que permite al cliente (el navegador o un programa Python) intercambiar mensajes con un servidor web. El cliente envía una petición y el servidor devuelve una respuesta. `https://` indica que el intercambio viaja por una conexión cifrada.

### La petición: qué se consulta

```text
GET /posts?userId=1 HTTP/1.1
Host: jsonplaceholder.typicode.com
```

Este ejemplo simplificado muestra el método `GET` (solicitar un recurso), el dominio y la ruta `/posts`. Lo que aparece tras `?` es la consulta: `userId=1` es un filtro que reconoce este servidor.

### La respuesta: qué devuelve el servidor

```text
HTTP/1.1 200 OK
Content-Type: application/json

[{"userId": 1, "id": 1, "title": "...", "body": "..."}]
```

| Parte | Qué indica | Cómo se consulta en Python |
| --- | --- | --- |
| Estado | `200`: la petición tuvo éxito. | `respuesta.status_code` |
| Encabezados | `Content-Type`: formato declarado del contenido. | `respuesta.headers.get("Content-Type")` |
| Cuerpo | Los datos recibidos. En el ejemplo se abrevia el listado. | `respuesta.text` para leerlo como texto. |

La línea en blanco separa encabezados y cuerpo. En hockey, el cuerpo es HTML (`text/html`); en la API de publicaciones, JSON (`application/json`). Un mismo método puede recibir formatos diferentes.

### Estados habituales

`200`: éxito. `403`: el servidor rechaza el acceso. `404`: recurso no encontrado. `429`: demasiadas peticiones. `500`: error del servidor. Si se agota el tiempo de espera, puede no llegar ninguna respuesta y, por tanto, ningún código HTTP.

En Python, `requests.get(...)` envía la petición. `raise_for_status()` comprueba si hubo un error HTTP (4xx o 5xx). Después se revisa el formato y se lee el cuerpo.

### Un 200 no garantiza que hayas recibido JSON

El código `200` indica que el servidor respondió con éxito; `Content-Type` indica qué formato declara. Si esperabas `/posts` pero acabaste en la página principal, puedes recibir `200` y `text/html`. Comprueba la URL final y el inicio del cuerpo. Si has recibido HTML, revisa la ruta antes de llamar a `.json()`: ese método interpreta JSON, no convierte HTML en JSON.

## B.2. Métodos HTTP y comprobación en Python

El método indica qué acción se solicita. Cada API documenta qué métodos admite en cada ruta.

| Método | Para qué sirve | Ejemplo |
| --- | --- | --- |
| `GET` | Consultar un recurso y recibir su contenido. | Leer la publicación 1. |
| `HEAD` | Consultar el estado y los encabezados, sin cuerpo. | Ver si un archivo es PDF. |
| `POST` | Enviar datos para que el servidor los procese. | Crear una publicación. |
| `PUT` | Crear o sustituir el recurso en la dirección indicada. | Sustituir una publicación. |
| `PATCH` | Aplicar cambios parciales a un recurso. | Cambiar solo el título. |
| `DELETE` | Solicitar la eliminación de un recurso. | Borrar la publicación 1. |

Con `HEAD` puedes ver `Content-Type` (formato) o `Content-Length` (tamaño en bytes), si se incluyen. `GET` ya trae encabezados, así que no hace falta un `HEAD` previo. `HEAD` no devuelve cuerpo para `.json()` y es distinto de la etiqueta `<head>` del HTML.

### Comprobar una respuesta antes de interpretarla como JSON

La función comprueba el estado y el formato declarado en las dos consultas. `.json()` comprueba después si el contenido es JSON válido.

In [ ]:
def comprobar_json(respuesta):
    respuesta.raise_for_status()
    cabecera = respuesta.headers.get("Content-Type", "")
    tipo = cabecera.split(";")[0].strip().lower()
    if tipo != "application/json":
        print("URL final:", respuesta.url)
        print("Inicio del cuerpo:", respuesta.text[:200])
        raise ValueError("La respuesta no declara application/json.")

Qué hace cada línea:

1. `def` define la función. El bloque indentado se ejecuta al llamarla.
2. `raise_for_status()` lanza un error si el estado HTTP es 4xx o 5xx.
3. `get(...)` busca el encabezado `Content-Type`. Si falta, devuelve la cadena vacía `""`.
4. `split(";")[0]` conserva el tipo antes del punto y coma. `strip()` quita espacios y `lower()` pasa a minúsculas.
5. `!=` significa «es distinto de». Si el tipo no coincide, se ejecutan las tres líneas siguientes.
6. `.url` muestra la dirección final. Puede diferir de la pedida si hubo una redirección.
7. `[:200]` muestra solo los primeros 200 caracteres. Si ves `<!DOCTYPE html>` o `<html>`, has recibido una página HTML.
8. `raise ValueError(...)` detiene la celda antes de llamar a `.json()`.

## C. JSON: del texto a Python

JSON es un formato de texto para intercambiar datos estructurados. Una API puede devolver JSON, CSV, XML u otros formatos. Consulta la documentación y comprueba `Content-Type` antes de elegir cómo leer la respuesta.

| En el JSON | En Python | Cómo se lee |
| --- | --- | --- |
| `{"id": 1}` | `dict` | Objeto: las llaves encierran pares de clave y valor. Los dos puntos los separan. |
| `[1, 2]` | `list` | Lista: los corchetes encierran elementos separados por comas. |
| `"Ejemplo"` | `str` | Cadena: el texto y las claves del objeto van entre comillas dobles. |
| `1`, `2.5` | `int`, `float` | Números. `"1"` entre comillas sería texto. |
| `true`, `false`, `null` | `True`, `False`, `None` | Booleanos y ausencia explícita de valor. En JSON se escriben en minúsculas. |

En `/posts` la estructura externa es una lista. Cada elemento es un objeto que representa una publicación. En `/posts/1` se recibe un único objeto. El código siguiente utiliza un registro recortado con un `null` para mostrar cómo se convierte, sin consultar la web.

In [ ]:
import json
texto = '[{"id": 1, "title": "Ejemplo", "body": null}]'
datos = json.loads(texto)
primera = datos[0]
print(primera["title"], primera["body"])

Qué hace cada línea:

1. `import json` carga el módulo de la biblioteca estándar que lee y escribe JSON.
2. Las comillas simples exteriores delimitan la cadena de Python. Dentro de ella, el JSON conserva sus comillas dobles y su valor `null`.
3. `json.loads(texto)` interpreta el texto. `datos` pasa a contener una lista de Python cuyo primer elemento es un diccionario.
4. `[0]` selecciona la primera publicación. `primera` guarda ese diccionario, no una cadena.
5. `["title"]` y `["body"]` recuperan los valores de esas claves. `print` muestra `Ejemplo None`: el `null` de JSON se convirtió en `None`.

Con una respuesta HTTP, `respuesta.json()` interpreta su cuerpo igual que `json.loads()` interpreta una cadena. Si el cuerpo contiene HTML, el método no lo convierte en JSON. En `/posts`, pandas convierte la lista resultante en una tabla: cada diccionario es una fila y sus claves son columnas.

### Comprueba la lectura

En el ejemplo, ¿qué tipo tiene `texto` antes de usar `json.loads()` y qué tipo tiene `datos` después? ¿Qué aparecería en pantalla si el campo `body` contuviera `"null"` entre comillas?

### ✏️ Respuesta

- `texto` es un `str` (texto), porque va entre comillas.
- `datos` es una `list` (lista) con un diccionario dentro, porque el JSON empieza por `[`.
- Si `body` fuera `"null"` con comillas, saldría `Ejemplo null`, porque sería un texto normal y no un valor vacío.

## 1. Scraping: localizar la tabla de hockey

[Abrir Scrape This Site: tabla de equipos de hockey](https://www.scrapethissite.com/pages/forms/)

La tabla de Scrape This Site muestra estadísticas de equipos de hockey por temporada. Una fila representa un equipo en un año; un mismo equipo puede aparecer en varias temporadas sin ser un duplicado.

### Qué mirar en el navegador

1. Abre la página y localiza los encabezados `Team Name`, `Year` y `Wins`.

2. Pulsa con el botón derecho sobre el nombre de un equipo y elige «Inspeccionar». Busca la celda `<td>`, la fila `<tr>` que la contiene y la tabla `<table>` a la que pertenece.

3. Localiza `Team Name` en el encabezado. El código utiliza ese texto para identificar la tabla.

### Fragmento simplificado de la tabla

```text
<table>
  <tr>
    <th>Team Name</th><th>Year</th><th>Wins</th>
  </tr>
  <tr>
    <td>Boston Bruins</td><td>1990</td><td>44</td>
  </tr>
</table>
```

| Etiqueta | Qué significa |
| --- | --- |
| `<table>` | Abre la tabla. `</table>` la cierra. |
| `<tr>` | Abre una fila. Cada fila agrupa varias celdas. |
| `<th>` | Marca una celda de encabezado, como el nombre de una columna. |
| `<td>` | Marca una celda de datos, como el año 1990. |

El fragmento muestra tres columnas de una fila. La tabla completa tiene más columnas y está paginada: cada consulta devuelve las filas de la página indicada en la URL.

Las reglas de `robots.txt` aplicables a esta ruta se explican en el apartado A.

## 2. Scraping: pedir y comprobar el HTML

Después de las importaciones, consulta la página y comprueba el estado y el formato de la respuesta antes de extraer la tabla.

In [ ]:
url_hockey = "https://www.scrapethissite.com/pages/forms/"
respuesta = requests.get(url_hockey, timeout=15)
respuesta.raise_for_status()
print(respuesta.status_code)
print(respuesta.headers.get("Content-Type"))
html = respuesta.text
print("<table" in html.lower())

Qué hace cada línea:

1. `url_hockey` guarda la dirección de la página como texto. Las comillas delimitan la cadena y `=` asigna ese valor al nombre.
2. `requests.get(...)` pide el recurso. `timeout=15` evita esperar indefinidamente a la conexión o a la llegada de datos. `respuesta` contiene el resultado HTTP completo.
3. `raise_for_status()` lanza un error si el servidor respondió con un código HTTP de error (4xx o 5xx). Así no intentamos leer una página de error como si fuera la tabla.
4. `.status_code` consulta el código de estado. `print(...)` lo muestra: `200` es el resultado esperado en esta consulta.
5. `.headers` contiene los metadatos de la respuesta. Aquí `.get("Content-Type")` busca un dato de ese conjunto; no hace otra petición web. El tipo esperado es `text/html`, quizá seguido de la codificación.
6. `.text` es el cuerpo de la respuesta convertido en texto. `html` contiene el código de la página, no un DataFrame.
7. `.lower()` crea una versión en minúsculas para buscar sin distinguir mayúsculas. `in` comprueba si aparece `<table` y devuelve `True` o `False`. Esto es una pista, no identifica qué tabla es.

### Qué esperar al ejecutar

```text
200
text/html; charset=utf-8
True
```

La codificación indicada en el encabezado puede variar. Un `200` indica que la petición HTTP ha tenido éxito. Comprueba también que la respuesta contiene la tabla buscada.

### Qué contienen las variables

| Variable | Contenido |
| --- | --- |
| `url_hockey` | Una cadena con la dirección. |
| `respuesta` | La respuesta HTTP: estado, encabezados y cuerpo. |
| `html` | Una cadena con el HTML que llegó en el cuerpo. |

## 3. Scraping: pasar la tabla a pandas

Usa `html` de la celda anterior. Pandas reconoce tablas HTML con filas y celdas. Para extraer datos de fichas de productos, el siguiente ejercicio usa Beautiful Soup.

In [ ]:
tablas = pd.read_html(StringIO(html), match="Team Name")
print(len(tablas))
hockey = tablas[0]
print(hockey.shape)
columnas = ["Team Name", "Year", "Wins"]
print(hockey[columnas].head(3).to_string(index=False))
print(hockey.dtypes)
print(hockey.isna().sum())

Qué hace cada línea:

1. `StringIO(html)` envuelve el texto para que se pueda leer como un archivo. `read_html(...)` devuelve una lista de DataFrames. `match="Team Name"` selecciona tablas cuyo texto incluye esa expresión.
2. `len(tablas)` indica cuántas tablas coinciden, no cuántos registros de hockey hay.
3. `[0]` toma la primera tabla de la lista. En este ejemplo buscamos la tabla de equipos. Si hubiera varias coincidencias, revisaríamos sus columnas antes de elegir.
4. `.shape` devuelve una pareja: (filas, columnas). Esta consulta incluye una sola página del sitio.
5. `columnas` contiene los tres nombres que se mostrarán. Deben coincidir con los encabezados originales.
6. `hockey[columnas]` elige esas columnas y `.head(3)` limita la vista a tres filas. `.to_string(index=False)` las imprime sin el índice. No se recorta el DataFrame original.
7. `.dtypes` muestra el tipo que pandas ha asignado a cada columna. Comprueba, por ejemplo, que `Wins` se haya leído como número.
8. `.isna()` señala los datos ausentes. `.sum()` cuenta esas señales por columna. Comprueba qué sucede con las celdas vacías de `OT Losses`.

### Primeras salidas de referencia

```text
1
(25, 9)
    Team Name  Year  Wins
Boston Bruins  1990    44
Buffalo Sabres 1990    31
Calgary Flames 1990    46
```

En la página consultada había 25 filas y 9 columnas en la tabla. La celda también muestra los tipos y el recuento de nulos que pandas detecta. Si el sitio cambia, usa las salidas de tu propia ejecución.

### Del scraping al EDA

En la columna `OT Losses` se observan huecos en la página consultada. Comprueba cómo aparecen en pandas y consulta su significado antes de decidir un tratamiento. Una celda vacía indica que no tenemos ahí un valor observado; no la conviertas directamente en cero.

## 3.1. Web scraping con Beautiful Soup

Extrae el nombre y el precio mostrado de hasta tres frutas del catálogo de Dia. Requests descarga la página. Beautiful Soup permite buscar las fichas y leer el texto de cada una.

[Abrir el catálogo de frutas de Dia](https://www.dia.es/frutas/c/L105)

[Consultar el robots.txt de Dia](https://www.dia.es/robots.txt)

Comprueba las reglas de `User-agent: *` para la ruta `/frutas/c/L105`. En el catálogo, pulsa con el botón derecho en el nombre de una fruta y elige «Inspeccionar». Localiza el atributo `data-test-id` del nombre, del precio y del bloque que los contiene.

### HTML simplificado de una ficha

```text
<div data-test-id="product-card">
  <a data-test-id="search-product-card-name">Caqui granel 500 g aprox.</a>
  <span data-test-id="search-product-card-unit-price">1,25 €</span>
  <span data-test-id="search-product-card-kilo-price">(2,49 €/KILO)</span>
</div>
```

El bloque exterior identificado como `product-card` reúne los datos de una fruta. Dentro hay dos importes distintos. En este ejemplo, 1,25 € es el precio mostrado para el formato de 500 g aprox. y 2,49 €/kg es el precio por kilo.

### Descarga la página

In [ ]:
url_frutas = "https://www.dia.es/frutas/c/L105"
respuesta_frutas = requests.get(url_frutas, timeout=20)
respuesta_frutas.raise_for_status()
html_frutas = respuesta_frutas.text

Qué hace cada línea:

1. `url_frutas` guarda la dirección de la categoría.
2. `requests.get()` pide la página. `timeout=20` limita la espera de la conexión o de los datos.
3. `raise_for_status()` detiene la celda si el servidor responde con un error HTTP.
4. `.text` recupera el HTML como una cadena y lo guarda en `html_frutas`.

La celda usa `requests` de la celda de importaciones. Si al importar Beautiful Soup aparece un error porque falta `bs4`, ejecuta `%pip install beautifulsoup4` en una celda de Colab y vuelve a intentarlo.

## 3.2. Ejercicio guiado: extraer nombre y precio

Ejecuta esta celda después de guardar el HTML en `html_frutas`. Fíjate en qué búsquedas se hacen sobre toda la página y cuáles se hacen dentro de una ficha.

In [ ]:
from bs4 import BeautifulSoup
pagina = BeautifulSoup(html_frutas, "html.parser")
fichas = pagina.select('[data-test-id="product-card"]')
print("Fichas encontradas:", len(fichas))
for ficha in fichas[:3]:
    nombre = ficha.select_one('[data-test-id="search-product-card-name"]')
    precio = ficha.select_one('[data-test-id="search-product-card-unit-price"]')
    if nombre is None or precio is None:
        continue
    producto = nombre.get_text(strip=True)
    importe = precio.get_text(strip=True)
    print(f"{producto} | {importe}")

Qué hace cada línea:

1. `from bs4 import BeautifulSoup` importa la herramienta que leerá el HTML.
2. `BeautifulSoup(..., "html.parser")` prepara el HTML para buscar en sus etiquetas. El segundo argumento indica qué analizador usar.
3. `select()` devuelve todas las fichas que coinciden con el selector. Los corchetes indican el atributo y su valor.
4. `len(fichas)` cuenta las fichas presentes en el HTML recibido.
5. `[:3]` toma hasta tres fichas y `for` las recorre una por una.
6. `select_one()` busca el primer elemento con ese atributo dentro de la ficha actual.
7. Repite la búsqueda dentro de la misma ficha para localizar el precio mostrado.
8. `None` indica que faltó el elemento buscado. `or` comprueba si falta alguno de los dos.
9. `continue` pasa a la ficha siguiente si faltaba el nombre o el precio.
10. `get_text(strip=True)` extrae el texto del nombre y quita los espacios de los extremos.
11. Extrae el texto del precio del mismo modo. El resultado conserva la coma decimal y el símbolo del euro.
12. `f"..."` inserta los valores de `producto` e `importe` en el texto que muestra `print()`.

### Comprueba tu salida

Compara la primera línea de producto con la ficha correspondiente en el navegador. El surtido y los precios pueden variar según el código postal y el momento. Si se encuentran cero fichas, comprueba si `product-card` aparece en `html_frutas`. El cuaderno incluye una muestra local para practicar la extracción.

### Amplía el ejercicio

Extrae el precio por kilo de cada ficha y muéstralo junto al nombre y al precio del producto. Busca el atributo correspondiente en el fragmento HTML del apartado 3.1. Explica qué precio usarías para comparar dos bandejas de pesos distintos.

### ✏️ Respuesta: precio por kilo

Añado una búsqueda más dentro de cada ficha, con el atributo del precio por kilo.

In [ ]:
for ficha in fichas[:3]:
    nombre = ficha.select_one('[data-test-id="search-product-card-name"]')
    precio = ficha.select_one('[data-test-id="search-product-card-unit-price"]')
    kilo = ficha.select_one('[data-test-id="search-product-card-kilo-price"]')
    if nombre is None or precio is None or kilo is None:
        continue
    print(nombre.get_text(strip=True), "|", precio.get_text(strip=True), "|", kilo.get_text(strip=True))

Para comparar dos bandejas de pesos distintos uso el **precio por kilo**, porque compara las dos con el mismo peso.

Ejemplo: 500 g por 2,00 € son 4,00 €/kg, y 1 kg por 3,50 € son 3,50 €/kg. La segunda es más barata aunque cueste más en total.

## 4. API: leer las rutas y el JSON

[Abrir JSONPlaceholder: listado de publicaciones](https://jsonplaceholder.typicode.com/posts)

Una API ofrece rutas para consultar recursos con una estructura definida. JSONPlaceholder devuelve publicaciones ficticias y no exige clave ni registro. Abre los enlaces y compara las respuestas.

| Ruta | Qué devuelve |
| --- | --- |
| `/posts` | Una lista de publicaciones. |
| `/posts/1` | Un objeto: la publicación cuyo identificador es 1. |
| `/posts?userId=1` | Una lista con las publicaciones del usuario 1. |

[Abrir una publicación: /posts/1](https://jsonplaceholder.typicode.com/posts/1)

[Abrir la lista filtrada: /posts?userId=1](https://jsonplaceholder.typicode.com/posts?userId=1)

### Cómo se lee la respuesta

```text
[
  {"userId": 1, "id": 1, "title": "...", "body": "..."},
  {"userId": 1, "id": 2, "title": "...", "body": "..."}
]
```

Es una muestra abreviada: los puntos suspensivos sustituyen al título y al texto. Los corchetes `[` y `]` delimitan la lista. Cada par de llaves `{` y `}` delimita una publicación. Los dos puntos unen un campo con su valor: `"userId": 1` significa que el autor tiene el identificador 1.

`id` identifica la publicación; `userId`, a su autor. Un usuario puede aparecer en varias filas porque puede escribir varias publicaciones. Aunque se representen con números, son identificadores, no medidas.

## 5. API: pasar el JSON a un DataFrame

La ruta `/posts` devuelve una lista de publicaciones. El método `.json()` interpreta el cuerpo de la respuesta; pandas convierte los registros en una tabla.

In [ ]:
url_posts = "https://jsonplaceholder.typicode.com/posts"
respuesta_api = requests.get(url_posts, timeout=15)
comprobar_json(respuesta_api)
print(respuesta_api.headers.get("Content-Type"))
registros = respuesta_api.json()
print(type(registros).__name__, len(registros))
posts = pd.DataFrame(registros)
print(posts.shape)
print(list(posts.columns))

Qué hace cada línea:

1. `url_posts` guarda la dirección del recurso `/posts`. Este endpoint devuelve una colección de publicaciones.
2. `requests.get(...)` hace una petición de lectura, igual que en el ejemplo de hockey. El servidor decide qué formato enviar.
3. `comprobar_json(...)` verifica el estado HTTP y el tipo declarado. La función se define en el apartado B.2.
4. El encabezado `Content-Type` indica el tipo de contenido. Esperamos `application/json`, con posibles datos adicionales de codificación.
5. `.json()` interpreta el texto JSON y lo convierte en objetos de Python. Aquí `registros` es una lista de diccionarios. Si el cuerpo no es JSON válido, esta línea falla.
6. `type(registros)` obtiene el tipo y `.__name__` su nombre, `list`. `len(registros)` cuenta los elementos de esa lista. La coma separa los dos valores que muestra `print`.
7. `pd.DataFrame(registros)` construye la tabla: un diccionario se convierte en una fila y sus claves en nombres de columnas.
8. `.shape` comprueba cuántas filas y columnas se han cargado.
9. `.columns` contiene los nombres de las columnas; `list(...)` los convierte en una lista fácil de leer.

### Salidas de referencia

```text
application/json; charset=utf-8
list 100
(100, 4)
['userId', 'id', 'title', 'body']
```

Comprueba qué significa una fila antes de contar: aquí contamos publicaciones, no usuarios. Verifica también si `id` identifica filas distintas y qué ocurre si falta algún campo. Si una clave falta en un registro, pandas puede dejar un dato ausente en la columna correspondiente.

## 6. API: pedir solo los registros que interesan

El parámetro `userId` permite filtrar las publicaciones en el servidor. La URL base es la misma que en la consulta anterior: `url_posts`.

In [ ]:
filtro = {"userId": 1}
respuesta_usuario = requests.get(url_posts, params=filtro, timeout=15)
comprobar_json(respuesta_usuario)
print(respuesta_usuario.url)
posts_usuario = pd.DataFrame(respuesta_usuario.json())
print(posts_usuario.shape)
print(posts_usuario["userId"].unique().tolist())

Qué hace cada línea:

1. `filtro` es un diccionario. La clave `"userId"` es el parámetro que reconoce la API; `1` es el identificador del usuario seleccionado.
2. `params=filtro` hace que Requests añada el filtro a la URL. En este caso el servidor devuelve las publicaciones del usuario indicado.
3. `comprobar_json(...)` detiene la celda si hay un error HTTP o si la respuesta no declara el formato esperado.
4. `.url` muestra la dirección usada, con `?userId=1`. El signo `?` separa la ruta de sus parámetros de consulta.
5. Primero `.json()` produce la lista filtrada y luego `DataFrame(...)` la convierte en tabla. Se guarda aparte, en `posts_usuario`.
6. `.shape` comprueba el tamaño de la tabla filtrada. No es el tamaño del conjunto original.
7. `posts_usuario["userId"]` selecciona la columna. `.unique()` obtiene sus valores distintos y `.tolist()` los convierte en una lista. El resultado esperado es `[1]`.

### Salidas de referencia

```text
https://jsonplaceholder.typicode.com/posts?userId=1
(10, 4)
[1]
```

El resultado sigue siendo una lista de publicaciones. Si filtras después de cargar el DataFrame, ya habrás descargado todos los registros. Aquí la consulta trae solo los que cumplen el filtro admitido por la API.

### Ejercicio: filtrar por otro usuario

Cambia el valor de `userId` de 1 a 2 y ejecuta de nuevo la celda. Comprueba que cambia la URL y verifica qué usuarios aparecen. Explica por qué `/posts/2` no responde a esa misma pregunta.

### ✏️ Respuesta: usuario 2

In [ ]:
filtro = {"userId": 2}
respuesta_usuario = requests.get(url_posts, params=filtro, timeout=15)
comprobar_json(respuesta_usuario)
print(respuesta_usuario.url)
posts_usuario = pd.DataFrame(respuesta_usuario.json())
print(posts_usuario.shape)
print(posts_usuario["userId"].unique().tolist())

- La URL cambia a `...posts?userId=2`.
- Solo aparece el usuario `[2]`.
- `/posts/2` no responde a la misma pregunta: da **una sola** publicación (la que tiene `id` 2), no las publicaciones del **autor** 2.

## 7. Ejercicios y comprobaciones

### Lectura de la tabla de hockey

Con la tabla de hockey, muestra las columnas `Team Name`, `Year` y `Goals For (GF)` en las primeras cinco filas. Indica qué significa una fila, qué columnas contienen recuentos y qué limitación tiene el hecho de haber consultado solo la primera página. Señala una columna que necesite revisión antes de analizarla.

### Consulta de publicaciones

Pide a JSONPlaceholder las publicaciones del usuario 2. Comprueba que el filtro funcionó y muestra las primeras tres filas de `id` y `title`. Abre `/posts/2` en el navegador y explica cómo cambia la estructura recibida y a quién identifica ese `2` en cada ruta.

### Si algo falla, ¿qué revisar?

| Síntoma | Qué comprobar |
| --- | --- |
| HTTP 404 | Comprueba que la ruta existe y está bien escrita. |
| HTTP 403 o 429 | Revisa si el sitio rechaza la petición o limita su frecuencia. No repitas la petición en bucle. |
| Error de tiempo de espera | Comprueba la conexión y si la página responde en el navegador. |
| No se encuentra la tabla | Revisa el encabezado usado en `match` y si la tabla existe en el HTML recibido. Lo que ves en pantalla puede haberse cargado después con JavaScript. |
| Error al interpretar JSON | Comprueba el estado, la ruta y `Content-Type`. Un servidor puede devolver HTML en vez del JSON esperado. |

### Documentación de consulta

[Estándar de robots.txt (RFC 9309)](https://www.rfc-editor.org/rfc/rfc9309.html)

[MDN: peticiones y respuestas HTTP](https://developer.mozilla.org/es/docs/Web/HTTP/Guides/Overview)

[MDN: métodos HTTP, incluido HEAD](https://developer.mozilla.org/es/docs/Web/HTTP/Reference/Methods)

[Estándar de JSON (RFC 8259)](https://www.rfc-editor.org/rfc/rfc8259.html)

[Guía de JSONPlaceholder: recursos y filtros](https://jsonplaceholder.typicode.com/guide/)

[pandas: lectura de tablas HTML con read_html](https://pandas.pydata.org/docs/reference/api/pandas.read_html.html)

[Requests: peticiones, respuestas y parámetros](https://requests.readthedocs.io/en/latest/user/quickstart/)

### ✏️ Respuesta: lectura de la tabla de hockey

In [ ]:
print(hockey[["Team Name", "Year", "Goals For (GF)"]].head(5))

- Una fila es **un equipo en un año**.
- Son recuentos: `Wins`, `Losses`, `OT Losses`, `Goals For (GF)` y `Goals Against (GA)`.
- Solo vemos la primera página, así que **faltan datos** y las conclusiones serían parciales.
- `OT Losses` necesita revisión: tiene huecos. No son ceros, son datos que faltan.

### ✏️ Respuesta: consulta de publicaciones

In [ ]:
filtro = {"userId": 2}
respuesta_usuario = requests.get(url_posts, params=filtro, timeout=15)
comprobar_json(respuesta_usuario)
posts_usuario = pd.DataFrame(respuesta_usuario.json())
print(posts_usuario["userId"].unique().tolist())
print(posts_usuario[["id", "title"]].head(3))

- El filtro funciona: solo aparece el usuario `[2]`.
- `/posts?userId=2` da una **lista** con las publicaciones del **usuario 2** (el `2` es el autor).
- `/posts/2` da **una sola** publicación, un objeto, la que tiene **`id` 2** (el `2` es la publicación).

## Lectura de muestras locales

Estas celdas leen texto guardado en el cuaderno. Puedes ejecutarlas directamente. Si te falta una biblioteca, consulta las indicaciones de instalación de cada apartado.

**Tabla de hockey con pandas.** La muestra conserva tres filas y cuatro columnas de la tabla. Comprueba el tamaño y observa cómo se representan las celdas vacías.

In [ ]:
from io import StringIO
import pandas as pd

html_muestra = """<table>
  <tr><th>Team Name</th><th>Year</th><th>Wins</th><th>OT Losses</th></tr>
  <tr><td>Boston Bruins</td><td>1990</td><td>44</td><td></td></tr>
  <tr><td>Buffalo Sabres</td><td>1990</td><td>31</td><td></td></tr>
  <tr><td>Calgary Flames</td><td>1990</td><td>46</td><td></td></tr>
</table>"""
equipos_demo = pd.read_html(StringIO(html_muestra), match="Team Name")[0]
print(equipos_demo)
print(equipos_demo["OT Losses"].isna().sum())

**Fichas de frutas con Beautiful Soup.** Esta celda aplica la extracción del ejercicio a dos fichas simplificadas. Los precios son una muestra fija. Puedes resolver aquí también la ampliación sobre el precio por kilo.

In [ ]:
html_frutas = """<div data-test-id="product-card">
  <a data-test-id="search-product-card-name">Caqui granel 500 g aprox.</a>
  <span data-test-id="search-product-card-unit-price">1,25 €</span>
  <span data-test-id="search-product-card-kilo-price">(2,49 €/KILO)</span>
</div>
<div data-test-id="product-card">
  <a data-test-id="search-product-card-name">Granada unidad 500 g aprox.</a>
  <span data-test-id="search-product-card-unit-price">1,50 €</span>
  <span data-test-id="search-product-card-kilo-price">(2,99 €/KILO)</span>
</div>"""
from bs4 import BeautifulSoup
pagina = BeautifulSoup(html_frutas, "html.parser")
fichas = pagina.select('[data-test-id="product-card"]')
print("Fichas encontradas:", len(fichas))
for ficha in fichas[:3]:
    nombre = ficha.select_one('[data-test-id="search-product-card-name"]')
    precio = ficha.select_one('[data-test-id="search-product-card-unit-price"]')
    if nombre is None or precio is None:
        continue
    producto = nombre.get_text(strip=True)
    importe = precio.get_text(strip=True)
    print(f"{producto} | {importe}")

**JSON.** Son tres registros ficticios con los mismos campos que `/posts`. `json.loads(...)` lee el JSON de la cadena en vez de una respuesta HTTP. En la última línea, `== 1` selecciona las filas del usuario 1. Este filtro se aplica en pandas: no se envía a un servidor.

In [ ]:
import json
import pandas as pd

json_muestra = """[
  {"userId": 1, "id": 1, "title": "Título de ejemplo A", "body": "Texto A"},
  {"userId": 1, "id": 2, "title": "Título de ejemplo B", "body": "Texto B"},
  {"userId": 2, "id": 11, "title": "Título de ejemplo C", "body": "Texto C"}
]"""
registros_demo = json.loads(json_muestra)
posts_demo = pd.DataFrame(registros_demo)
print(posts_demo[["userId", "id", "title"]])
print(posts_demo.loc[posts_demo["userId"] == 1, ["userId", "id"]])